# PubTabNet — General Table Agent Eval

Benchmarks `general_table` agent against 20 PubTabNet sample images.
Ground truth comes from PubTabNet's HTML annotations, converted to DataFrames via `pd.read_html()`.

In [25]:
import sys, os, json
from io import StringIO
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from PIL import Image
from shared.client import client, DEFAULT_MODEL
from shared.eval import compare_tables, print_accuracy_summary
from agents.general_table.extract import extract_tables_from_page
from pubtabnet.examples.utils import format_html

In [26]:
MODEL = DEFAULT_MODEL
EXAMPLES_DIR = os.path.join("..", "pubtabnet", "examples")
JSONL_PATH = os.path.join(EXAMPLES_DIR, "PubTabNet_Examples.jsonl")

# Load all 20 examples
examples = []
with open(JSONL_PATH) as f:
    for line in f:
        examples.append(json.loads(line))

print(f"Loaded {len(examples)} PubTabNet examples")

Loaded 20 PubTabNet examples


In [27]:
# Convert HTML ground truth to DataFrames
ground_truths = {}
skipped = []

for ex in examples:
    fname = ex["filename"]
    try:
        html_str = format_html(ex)
        dfs = pd.read_html(StringIO(html_str))
        if not dfs:
            skipped.append((fname, "pd.read_html returned no tables"))
            continue
        df = dfs[0]
        # Flatten multi-level columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [' '.join(str(c) for c in col).strip() for col in df.columns]
        ground_truths[fname] = df
    except Exception as e:
        skipped.append((fname, str(e)))

print(f"Valid ground truths: {len(ground_truths)}/{len(examples)}")
if skipped:
    print(f"Skipped: {skipped}")

Valid ground truths: 20/20


/Users/roshgill/Desktop/HumaAI/pubtabnet/examples/utils.py:31: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 31 of the file /Users/roshgill/Desktop/HumaAI/pubtabnet/examples/utils.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = bs(html_string)
/Users/roshgill/Desktop/HumaAI/pubtabnet/examples/utils.py:31: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

Th

In [28]:
# Run extraction and compare (single image test)
from IPython.display import display, HTML

results = []
test_items = list(ground_truths.items())[:1]  # Just first image for now

for fname, df_truth in test_items:
    img_path = os.path.join(EXAMPLES_DIR, fname)
    if not os.path.exists(img_path):
        print(f"  Image not found: {img_path}, skipping")
        continue

    print(f"\n--- {fname} ({len(df_truth)} rows, {len(df_truth.columns)} cols) ---")

    image = Image.open(img_path)

    try:
        page_result = extract_tables_from_page(client, MODEL, image)
    except Exception as e:
        print(f"  API error: {e}")
        results.append({"filename": fname, "total_cells": 0, "correct_cells": 0})
        continue

    # Debug: show what Gemini returned
    print(f"  Segments returned: {len(page_result.tables)}")
    for i, seg in enumerate(page_result.tables):
        print(f"    seg[{i}]: headers={seg.headers!r} rows={len(seg.rows)}")

    # Build DataFrames directly from segments (single-image, no multi-page stitching)
    dfs = []
    for seg in page_result.tables:
        if not seg.rows:
            continue
        if seg.headers:
            dfs.append(pd.DataFrame(seg.rows, columns=seg.headers))
        else:
            # No headers — use positional columns
            dfs.append(pd.DataFrame(seg.rows))

    if not dfs:
        print("  No tables extracted!")
        results.append({"filename": fname, "total_cells": 0, "correct_cells": 0})
        continue

    df_extracted = dfs[0]

    # Show tables side by side
    display(HTML(
        '<div style="display:flex; gap:2em;">'
        '<div><h4>Ground Truth</h4>' + df_truth.to_html(index=False) + '</div>'
        '<div><h4>Extracted</h4>' + df_extracted.to_html(index=False) + '</div>'
        '</div>'
    ))

    result = compare_tables(df_truth, df_extracted)
    result["filename"] = fname
    results.append(result)
    print_accuracy_summary(result)


--- PMC4840965_004_00.png (27 rows, 4 cols) ---
  Segments returned: 1
    seg[0]: headers=['Variable', 'Hazard ratio', '95 % CI', 'p value*'] rows=27


Variable,Hazard ratio,95 % CI,p value*
Age (median),NaN,NaN,0.716
≤69,1.000,NaN,NaN
>69,0.839,0.310–2.268,NaN
Gender,NaN,NaN,0.142
Male,1.000,NaN,NaN
Female,0.426,0.152–1.190,NaN
Type of surgery,NaN,NaN,0.010
Low anterior resection,1.000,NaN,NaN
Abdominoperineal resection,3.140,0.919–10.725,NaN
Tumor location,NaN,NaN,0.710


ACCURACY SUMMARY
Ground truth rows:  27
Extracted rows:     27
Rows compared:      27
Missing rows:       0
Extra rows:         0

Total cells compared: 108
Correct cells:        97
Cell accuracy:        89.8%
Row match:            59.3%  (16/27 rows fully correct)

DETAILED MISMATCHES (11 cells)


,Row,Column,Ground Truth,Extracted
0,1,hazard ratio,10,1000
1,4,hazard ratio,10,1000
2,7,hazard ratio,10,1000
3,8,hazard ratio,314,3140
4,10,hazard ratio,10,1000
5,14,hazard ratio,10,1000
6,18,hazard ratio,10,1000
7,21,hazard ratio,10,1000
8,25,hazard ratio,10,1000
9,6,p value*,001,0010



PER-COLUMN ACCURACY


,Column,Total Cells,Correct,Errors,Accuracy
0,variable,27,27,0,100.0%
1,hazard ratio,27,18,9,66.7%
2,95 % ci,27,27,0,100.0%
3,p value*,27,25,2,92.6%


In [29]:
# Aggregate accuracy
evaluated = [r for r in results if r.get("total_cells", 0) > 0]
if evaluated:
    total_cells = sum(r["total_cells"] for r in evaluated)
    correct_cells = sum(r["correct_cells"] for r in evaluated)
    total_rows_compared = sum(r["rows_compared"] for r in evaluated)
    total_rows_correct = sum(r["rows_fully_correct"] for r in evaluated)
    print(f"{'=' * 60}")
    print(f"AGGREGATE ACROSS {len(evaluated)} TABLES")
    print(f"{'=' * 60}")
    print(f"Cell accuracy:  {correct_cells}/{total_cells} = {correct_cells/total_cells:.1%}")
    print(f"Row match:      {total_rows_correct}/{total_rows_compared} = {total_rows_correct/total_rows_compared:.1%}")
    print(f"Tables with no extraction: {len(results) - len(evaluated)}")
else:
    print("No tables were evaluated.")

AGGREGATE ACROSS 1 TABLES
Cell accuracy:  97/108 = 89.8%
Row match:      16/27 = 59.3%
Tables with no extraction: 0
